In [1]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP & IMPORTS
# ==========================================
# Run this cell first to install required packages and import dependencies.

# Install specialized packages not pre-loaded on Kaggle
# Note: mamba-ssm is excluded due to Kaggle C++ compilation issues; the model will fallback to LSTM.
!pip install -q tifffile spectral

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import random
import joblib

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from einops import rearrange

# Explainable AI
import shap

# Kaggle Output Directory
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set random seed for reproducibility (will be dynamically changed in 10-seed loop)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print("✅ Cell 1 Complete: Environment configured and libraries imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/249.0 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 kB 5.2 MB/s eta 0:00:00


✅ Cell 1 Complete: Environment configured and libraries imported.


In [2]:
# ==========================================
# CELL 2: DATASET EXTRACTION & PREPROCESSING
# ==========================================

# 1. Reverse-Engineer Local Dataset (.bil files)
def read_bil_file(filepath, samples=320, bands=168, dtype=np.uint16):
    filesize = os.path.getsize(filepath)
    bytes_per_pixel = np.dtype(dtype).itemsize
    lines = filesize / (samples * bands * bytes_per_pixel)
    if not lines.is_integer():
        return None
    lines = int(lines)
    raw_data = np.fromfile(filepath, dtype=dtype)
    img_cube = raw_data.reshape((lines, bands, samples))
    img_cube = np.transpose(img_cube, (0, 2, 1)) # (H, W, C)
    return img_cube

def extract_patches(img_cube, label, condition, patch_size=15):
    h, w, c = img_cube.shape
    patches, labels, conditions = [], [], []
    for i in range(0, h - patch_size, patch_size):
        for j in range(0, w - patch_size, patch_size):
            patch = img_cube[i:i+patch_size, j:j+patch_size, :]
            patches.append(patch)
            labels.append(label)
            conditions.append(condition) # 0=Dry, 1=Moist
    return patches, labels, conditions

def load_local_dataset(base_dir):
    print(f"⏳ Extracting Local Dataset from: {base_dir}")
    X, y, cond = [], [], []
    
    # Define mapping based on user directory structure
    class_map = {'Black Soil': 0, 'Red Soil': 1, 'Yellow Soil': 2}
    
    bil_files = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.bil'):
                bil_files.append(os.path.join(root, f))
                
    if len(bil_files) == 0:
        raise FileNotFoundError("❌ CRITICAL: 0 .bil files found! You MUST click 'Add Data' -> 'Your Datasets' -> 'local-soil-hyperspectral-dataset' in your Kaggle Notebook to mount the local dataset!")
        
    print(f"Found {len(bil_files)} .bil files. Extracting patches...")
    
    for filepath in bil_files:
        root = os.path.dirname(filepath)
        img = read_bil_file(filepath)
        if img is None: continue
        
        # Assign labels
        label = -1
        if 'black' in root.lower(): label = 0
        elif 'red' in root.lower(): label = 1
        elif 'yellow' in root.lower(): label = 2
        
        condition = 1 if 'moist' in root.lower() else 0
        
        if label != -1:
            p, l, c = extract_patches(img, label, condition)
            X.extend(p)
            y.extend(l)
            cond.extend(c)
                    
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    cond = np.array(cond)
    print(f"✅ Local Data Loaded: {X.shape[0]} patches extracted. Shape: {X.shape}")
    return X, y, cond

import scipy.io as sio

# 2. Real Public Dataset Loaders
# This explicitly loads the 4 public datasets from Kaggle's input directory.
def load_mat_dataset(filepath, data_key, label_key):
    """Generic loader for .mat hyperspectral datasets."""
    try:
        mat = sio.loadmat(filepath)
        X = mat[data_key]
        y = mat[label_key].flatten()
        return X, y
    except Exception as e:
        print(f"Failed to load {filepath}: {e}")
        return None, None

def load_karly_csv(filepath):
    """Specific loader for the KarLy soil moisture .csv dataset on Kaggle."""
    try:
        import pandas as pd
        df = pd.read_csv(filepath)
        # Assuming bands are the numeric columns and moisture is a target
        band_cols = [c for c in df.columns if c.replace('.', '').isdigit()]
        X = df[band_cols].values
        # Binarize moisture for zero-shot testing (dry vs moist)
        y = (df['soil_moisture'] > df['soil_moisture'].median()).astype(int).values
        return X, y
    except Exception as e:
        print(f"Failed to load KarLy CSV at {filepath}: {e}")
        return None, None

class HyBEARLazyDataset(Dataset):
    def __init__(self, metadata_csv, root_dir):
        import pandas as pd
        self.df = pd.read_csv(metadata_csv)
        self.root_dir = root_dir
        
        print("⏳ Building robust image index for Kaggle filesystem...")
        # os.walk is mathematically guaranteed to find all files. followlinks=True fixes Kaggle symlink bugs.
        self.image_index = {}
        for root, dirs, files in os.walk(self.root_dir, followlinks=True):
            for f in files:
                # Case-insensitive match! Kaggle users often upload .TIFF instead of .tiff
                if f.lower().endswith(('.tif', '.tiff')):
                    # Strip extension so it matches regardless of .tif vs .tiff
                    basename = os.path.splitext(f)[0].strip()
                    self.image_index[basename] = os.path.join(root, f)
            
            
        # Filter out metadata rows where the physical file wasn't uploaded
        self.df['basename'] = self.df['IMAGE'].apply(lambda x: os.path.splitext(str(x).strip())[0])
        original_len = len(self.df)
        self.df = self.df[self.df['basename'].isin(self.image_index.keys())].reset_index(drop=True)
        
        print(f"✅ Fast index built! Found {len(self.image_index)} physical images.")
        print(f"🧹 Metadata Cleaned: Synced {len(self.df)} physical files out of {original_len} total CSV rows!")
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        import tifffile
        row = self.df.iloc[idx]
        img_name = row['IMAGE']
        
        # Strip extension and hidden whitespaces from the requested name
        basename = os.path.splitext(img_name.strip())[0]
        
        # O(1) Absolute Path Lookup
        img_path = self.image_index.get(basename)
        
        if img_path is None:
            if getattr(self, 'error_count', 0) < 100:
                print(f"🚨 CRITICAL PATH ERROR: {img_name} NOT FOUND ANYWHERE IN {self.root_dir}")
                self.error_count = getattr(self, 'error_count', 0) + 1
            elif getattr(self, 'error_count', 0) == 100:
                print(f"🚨 (Suppressing further path errors to prevent log spam...)")
                self.error_count = getattr(self, 'error_count', 0) + 1
            spectrum = np.zeros(250)
        else:
            try:
                img = tifffile.imread(img_path)
                # Take spatial average to get a 1D spectrum for the patch
                if img.ndim == 3:
                    spectrum = img.mean(axis=(0,1)) if img.shape[0] > img.shape[2] else img.mean(axis=(1,2))
                else:
                    spectrum = img
            except Exception as e:
                print(f"🚨 CRITICAL READ ERROR: Corrupt TIFF {img_path}! ({e})")
                spectrum = np.zeros(250)
            
        label = 1 if row.get('SOIL', 0) > 0 else 0
        return torch.tensor(spectrum, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

class TintoLazyDataset(Dataset):
    def __init__(self, hdr_path, labels_hdr_path=None):
        import spectral.io.envi as envi
        try:
            # We bypass the Hylite collection header and directly load the VNIR .dat sensor array
            bin_path = hdr_path.replace('.hdr', '.dat')
            self.img = envi.open(hdr_path, image=bin_path).open_memmap()
            
            if labels_hdr_path:
                l_bin_path = labels_hdr_path.replace('.hdr', '.dat')
                self.labels = envi.open(labels_hdr_path, image=l_bin_path).open_memmap()
            else:
                self.labels = None
                
            self.h, self.w, self.bands = self.img.shape
            self.length = self.h * self.w
            self.is_2d = True
        except Exception as e:
            raise RuntimeError(f"Tinto ENVI load failed: {e}")
            
    def __len__(self):
        return self.length
        
    def __getitem__(self, idx):
        r = idx // self.w
        c = idx % self.w
        spectrum = self.img[r, c, :]
        label = self.labels[r, c, 0] if self.labels is not None else 0
        return torch.tensor(spectrum, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

def load_public_datasets(kaggle_input_dir="/kaggle/input"):
    print("⏳ Loading REAL Public Datasets (HyBEAR, KarLy, Shangyu, Tinto)...")
    public_data = {}
    
    # 1. HyBEAR (Pre-training Dataset)
    hybear_dir = os.path.join(kaggle_input_dir, "dev123123456/hybear-mini-dataset") if not os.path.exists(os.path.join(kaggle_input_dir, "datasets/dev123123456/hybear-mini-dataset")) else os.path.join(kaggle_input_dir, "datasets/dev123123456/hybear-mini-dataset")
    hybear_csv = glob.glob(os.path.join(hybear_dir, "**", "metadata.csv"), recursive=True) or glob.glob(os.path.join(kaggle_input_dir, "**", "metadata.csv"), recursive=True)
    if hybear_csv:
        # Pass the directory containing the csv as the root for images
        root_dir = os.path.dirname(hybear_csv[0])
        public_data['HyBEAR'] = {'LazyDataset': HyBEARLazyDataset(hybear_csv[0], root_dir)}
        print(f"✅ HyBEAR LazyLoader Initialized")
    else:
        print("⚠️ HyBEAR metadata.csv not found in Kaggle inputs.")

    # 2. KarLy (Zero-Shot Moisture Test) from CSV
    karly_dir = os.path.join(kaggle_input_dir, "binaryjoker/hyperspectral-benchmark-dataset-on-soil-moisture") if not os.path.exists(os.path.join(kaggle_input_dir, "datasets/binaryjoker/hyperspectral-benchmark-dataset-on-soil-moisture")) else os.path.join(kaggle_input_dir, "datasets/binaryjoker/hyperspectral-benchmark-dataset-on-soil-moisture")
    karly_path = glob.glob(os.path.join(karly_dir, "**", "soilmoisture_dataset.csv"), recursive=True) or glob.glob(os.path.join(kaggle_input_dir, "**", "soilmoisture_dataset.csv"), recursive=True)
    if karly_path:
        X, y = load_karly_csv(karly_path[0])
        if X is not None:
            public_data['KarLy'] = {'X_test': X, 'y_test': y}
            print(f"✅ KarLy Loaded: {X.shape}")
    else:
        print("⚠️ KarLy CSV not found in Kaggle inputs.")

    # 3. Indian Pines (Zero-Shot Agriculture/Soil Test)
    ip_dir = os.path.join(kaggle_input_dir, "abhijeetgo/indian-pines-hyperspectral-dataset") if not os.path.exists(os.path.join(kaggle_input_dir, "datasets/abhijeetgo/indian-pines-hyperspectral-dataset")) else os.path.join(kaggle_input_dir, "datasets/abhijeetgo/indian-pines-hyperspectral-dataset")
    indian_pines_path = glob.glob(os.path.join(ip_dir, "**", "indianpinearray.npy"), recursive=True) or glob.glob(os.path.join(kaggle_input_dir, "**", "indianpinearray.npy"), recursive=True)
    ip_gt_path = glob.glob(os.path.join(ip_dir, "**", "IPgt.npy"), recursive=True) or glob.glob(os.path.join(kaggle_input_dir, "**", "IPgt.npy"), recursive=True)
    if indian_pines_path and ip_gt_path:
        try:
            X_ip = np.load(indian_pines_path[0])
            y_ip = np.load(ip_gt_path[0]).flatten()
            
            # Flatten X_ip to (N, Bands) and remove background pixels (label == 0)
            X_ip = X_ip.reshape(-1, X_ip.shape[-1])
            valid_idx = np.where(y_ip > 0)[0]
            X_ip = X_ip[valid_idx]
            y_ip = y_ip[valid_idx] - 1  # 1-16 -> 0-15
            
            public_data['IndianPines'] = {'X_test': X_ip, 'y_test': y_ip}
            print(f"✅ Indian Pines Loaded: {X_ip.shape}")
        except Exception as e:
            print(f"Failed to load Indian Pines: {e}")
    else:
        print("⚠️ Indian Pines not found in Kaggle inputs.")

    # 4. Tinto (Zero-Shot Mineral Test)
    tinto_dir = os.path.join(kaggle_input_dir, "dev123123456/tinto-hyperspectral-digital-outcrop") if not os.path.exists(os.path.join(kaggle_input_dir, "datasets/dev123123456/tinto-hyperspectral-digital-outcrop")) else os.path.join(kaggle_input_dir, "datasets/dev123123456/tinto-hyperspectral-digital-outcrop")
    
    # We bypass the 'real.hdr' Hylite Collection header and directly target the 2D VNIR sensor
    tinto_real_hdr = os.path.join(tinto_dir, "Tinto/tinto2D.hyc/view1.hyc/real.hyc/vnir.hdr")
    tinto_labels_hdr = os.path.join(tinto_dir, "Tinto/tinto2D.hyc/view1.hyc/labels_basic.hdr")
    
    if os.path.exists(tinto_real_hdr) and os.path.exists(tinto_labels_hdr):
        try:
            public_data['Tinto'] = {'LazyDataset': TintoLazyDataset(tinto_real_hdr, tinto_labels_hdr)}
            print(f"✅ Tinto LazyLoader Initialized")
        except Exception as e:
            print(f"⚠️ Tinto failed to load: {e}")
    else:
        print("⚠️ Tinto VNIR 2D files not found in Kaggle inputs.")
        
    return public_data

class SoilDataset(Dataset):
    def __init__(self, X, y, indices=None):
        self.X = X
        self.y = y
        self.indices = indices if indices is not None else np.arange(len(X))
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x = torch.as_tensor(self.X[real_idx], dtype=torch.float32)
        # Normalize spectra (Standard Normal Variate) on the fly
        mean = x.mean()
        std = x.std() + 1e-8
        x = (x - mean) / std
        y_val = torch.as_tensor(self.y[real_idx], dtype=torch.long)
        return x, y_val

print("✅ Cell 2 Complete: Data pipeline ready.")


✅ Cell 2 Complete: Data pipeline ready.


In [3]:
# ==========================================
# CELL 3: MODEL ARCHITECTURES (6 FAMILIES)
# ==========================================

# 1. 1D-CNN (Basic Deep Learning)
class CNN1D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(bands, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.fc = nn.Linear(128, classes)
        
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :] # Shape (B, C)
        x = x.unsqueeze(2) # (B, C, 1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.squeeze(2)
        return self.fc(x)

# 2. 3D-CNN (Spatial-Spectral)
class CNN3D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        # Input: (B, C, Depth=bands, H, W)
        self.conv1 = nn.Conv3d(1, 8, kernel_size=(7, 3, 3), padding=(0,1,1))
        self.conv2 = nn.Conv3d(8, 16, kernel_size=(5, 3, 3), padding=(0,1,1))
        self.fc = nn.Linear(16 * (bands - 10) * 15 * 15, classes)
        
    def forward(self, x):
        # x: (B, H, W, C) -> (B, 1, C, H, W)
        x = x.permute(0, 3, 1, 2).unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.flatten(1)
        return self.fc(x)

# 3. 1D-ResNet
class ResNet1DBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
    def forward(self, x):
        return F.relu(self.conv2(F.relu(self.conv1(x))) + x)

class ResNet1D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        self.conv_in = nn.Conv1d(bands, 64, kernel_size=3, padding=1)
        self.res1 = ResNet1DBlock(64)
        self.res2 = ResNet1DBlock(64)
        self.fc = nn.Linear(64, classes)
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(2)
        x = F.relu(self.conv_in(x))
        x = self.res2(self.res1(x)).squeeze(2)
        return self.fc(x)

# 4. SOTA: Vision Transformer (ViT) / Self-Attention
class ViT1D(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128, depth=4, heads=4):
        super().__init__()
        self.patch_embed = nn.Linear(1, dim)
        self.pos_embed = nn.Parameter(torch.randn(1, bands, dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.fc = nn.Linear(dim, classes)
        
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(-1) # (B, Bands, 1)
        x = self.patch_embed(x) + self.pos_embed
        x = self.transformer(x)
        x = x.mean(dim=1) # Global Average Pooling
        return self.fc(x)

# 5. Graph Convolutional Network (GCN) Stub
class GCN1D(nn.Module):
    def __init__(self, bands=168, classes=3):
        super().__init__()
        # A simple spectral GCN approximation using Conv1D over spectral graphs
        self.conv1 = nn.Conv1d(bands, 64, kernel_size=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=1)
        self.fc = nn.Linear(128, classes)
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.fc(x.squeeze(2))

# 6. Mamba State-Space Model
class Mamba1D(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128):
        super().__init__()
        self.proj = nn.Linear(1, dim)
        try:
            from mamba_ssm import Mamba
            self.mamba = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
            self.is_lstm = False
        except ImportError:
            # Fallback to simple RNN/LSTM if mamba fails to import on Kaggle
            self.mamba = nn.LSTM(dim, dim, batch_first=True)
            self.is_lstm = True
        self.fc = nn.Linear(dim, classes)
        
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = x.unsqueeze(-1)
        x = self.proj(x)
        if self.is_lstm:
            x, _ = self.mamba(x)
        else:
            x = self.mamba(x)
        x = x.mean(dim=1)
        return self.fc(x)

# 7. Ablation Model: ViT Without Attention (Just an MLP on the patches)
class ViT_No_Attention(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128):
        super().__init__()
        self.fc1 = nn.Linear(bands, dim)
        self.fc2 = nn.Linear(dim, classes)
    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h//2, w//2, :]
        x = F.relu(self.fc1(x))
        return self.fc2(x)

print("✅ Cell 3 Complete: Architectures defined.")


✅ Cell 3 Complete: Architectures defined.


In [4]:
# ==========================================
# CELL 4: 10-SEED TRAINING LOOP
# ==========================================

def train_sklearn_model(model_name, X, y, train_idx, test_idx):
    if X.ndim == 4:
        b, h, w, c = X.shape
        X_train_flat = X[train_idx, h//2, w//2, :]
        X_test_flat = X[test_idx, h//2, w//2, :]
    else:
        X_train_flat = X[train_idx]
        X_test_flat = X[test_idx]
        
    y_train = y[train_idx]
    y_test = y[test_idx]
    
    if model_name == 'SVM': clf = SVC(kernel='rbf', class_weight='balanced')
    elif model_name == 'MLP': clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
    elif model_name == 'XGBoost': clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
    elif model_name == 'LGBM': clf = lgb.LGBMClassifier(class_weight='balanced')
    else: raise ValueError("Unknown model")
    
    clf.fit(X_train_flat, y_train)
    preds = clf.predict(X_test_flat)
    acc = accuracy_score(y_test, preds)
    kappa = cohen_kappa_score(y_test, preds)
    return acc, kappa, preds, clf

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        return focal_loss.mean()

def train_pytorch_model(model, train_loader, test_loader, epochs=100, patience=10, class_weights=None):
    import copy
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    if class_weights is not None:
        weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
        criterion = FocalLoss(alpha=weights_tensor)
    else:
        # Standard loss is perfect for perfectly balanced 50/50 Sampler batches
        criterion = nn.CrossEntropyLoss()
        
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    losses = []
    
    best_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        train_loss = epoch_loss / len(train_loader)
        losses.append(train_loss)
        
        # Validation for early stopping
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                out = model(X_batch.to(device))
                val_loss += criterion(out, y_batch.to(device)).item()
        val_loss /= len(test_loader)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"      Early stopping triggered at epoch {epoch+1}")
            break
            
    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
            
    # Evaluate
    model.eval()
    all_preds, all_y = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            out = model(X_batch.to(device))
            preds = torch.argmax(out, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_y.extend(y_batch.numpy())
            
    acc = accuracy_score(all_y, all_preds)
    kappa = cohen_kappa_score(all_y, all_preds)
    return acc, kappa, np.array(all_preds), model, losses

def execute_10_seed_training(X, y=None, conditions=None, dataset_name="Dataset", bands=168, classes=3, num_seeds=3, train_size=0.8, test_size=0.2, epochs=100, patience=20):
    print(f"\n⏳ Starting {num_seeds}-Seed Rigorous Training Pipeline for {dataset_name}...")
    is_lazy = isinstance(X, Dataset)
    
    # Exclude Sklearn models for lazy datasets, and 3D models for 1D datasets
    if is_lazy:
        models_to_test = ['1D-CNN', 'GCN', 'Mamba', 'ViT']
    elif X.ndim < 4:
        models_to_test = ['SVM', 'LGBM', 'XGBoost', '1D-CNN', 'GCN', 'Mamba', 'ViT']
    else:
        models_to_test = ['SVM', 'LGBM', 'XGBoost', '1D-CNN', '3D-CNN', 'GCN', 'Mamba', 'ViT']
        
    results = {m: {'acc': [], 'kappa': []} for m in models_to_test}
    
    from sklearn.utils.class_weight import compute_class_weight
    
    for seed in range(num_seeds):
        seed_everything(seed)
        print(f"\n--- Running Seed {seed+1}/{num_seeds} ---")
        
        if is_lazy:
            # Extract lazy labels for stratify & weighting
            if hasattr(X, 'df'):
                y_all = (X.df['SOIL'] > 0).astype(int).values
            elif hasattr(X, 'labels') and X.labels is not None:
                y_all = X.labels[:, :, 0].flatten()
            else:
                y_all = None

            indices = list(range(len(X)))
            
            # Auto-Scale Split Size: Prevent starvation on tiny datasets, prevent timeouts on massive datasets
            dynamic_train_size = 0.8 if len(X) < 5000 else 0.05
            print(f"📊 Auto-Scaling Split: Using {dynamic_train_size*100:.1f}% for training ({int(len(X) * dynamic_train_size)} samples)")
            
            # Stratified Split
            if y_all is not None:
                # Grab EXACTLY train_size (e.g. 80%) and perfectly stratify
                train_idx, temp_idx, y_train_weights, temp_y = train_test_split(indices, y_all, train_size=train_size, stratify=y_all, random_state=seed)
                # Grab EXACTLY test_size of the total length from the remainder
                test_ratio = min(1.0, test_size / (1.0 - dynamic_train_size))
                if test_ratio == 1.0:
                    test_idx = temp_idx
                else:
                    test_idx, _, _, _ = train_test_split(temp_idx, temp_y, train_size=test_ratio, stratify=temp_y, random_state=seed)
            else:
                np.random.shuffle(indices)
                tr_split = int(train_size * len(X))
                te_split = tr_split + int(test_size * len(X))
                train_idx = indices[:tr_split]
                test_idx = indices[tr_split:te_split]
                y_train_weights = None
            
            # Bulletproof Sampler: guarantees 50/50 batches regardless of data volume
            if y_train_weights is not None:
                unique_classes, counts = np.unique(y_train_weights, return_counts=True)
                # Ensure counts avoids divide by zero
                weight_dict = {cls: (1.0 / max(c, 1)) for cls, c in zip(unique_classes, counts)}
                sample_weights = [weight_dict[y] for y in y_train_weights]
                sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
                train_loader = DataLoader(torch.utils.data.Subset(X, train_idx), batch_size=16, sampler=sampler)
            else:
                train_loader = DataLoader(torch.utils.data.Subset(X, train_idx), batch_size=16, shuffle=True)
                
            test_loader = DataLoader(torch.utils.data.Subset(X, test_idx), batch_size=16, shuffle=False)
            X_train, y_train, X_test, y_test = None, None, None, None
        else:
            # Split: Train on Dry (condition 0) + 10% Moist. Test exclusively on remaining Moist.
            moist_indices = np.where(conditions == 1)[0]
            dry_indices = np.where(conditions == 0)[0]
            
            train_moist, test_moist = train_test_split(moist_indices, test_size=test_size, random_state=seed)
            train_idx = np.concatenate([dry_indices, train_moist])
            test_idx = test_moist
            
            # PyTorch Loaders (Zero-Copy)
            train_loader = DataLoader(SoilDataset(X, y, train_idx), batch_size=32, shuffle=True)
            test_loader = DataLoader(SoilDataset(X, y, test_idx), batch_size=32, shuffle=False)
            y_train_weights = y[train_idx]
            
        # Calculate Class Weights
        if y_train_weights is not None:
            unique_classes = np.unique(y_train_weights)
            c_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_weights)
            # Ensure weight array covers all expected classes
            full_weights = np.ones(classes)
            for i, cls in enumerate(unique_classes):
                if cls < classes:
                    full_weights[cls] = c_weights[i]
            c_weights = full_weights
        else:
            c_weights = None
        
        for m in models_to_test:
            if m in ['SVM', 'LGBM', 'XGBoost', 'MLP'] and not is_lazy:
                acc, kap, preds, _ = train_sklearn_model(m, X, y, train_idx, test_idx)
            else:
                if m == '1D-CNN': net = CNN1D(bands=bands, classes=classes)
                elif m == '3D-CNN': net = CNN3D(bands=bands, classes=classes)
                elif m == 'GCN': net = GCN1D(bands=bands, classes=classes)
                elif m == 'Mamba': net = Mamba1D(bands=bands, classes=classes)
                elif m == 'ViT': net = ViT1D(bands=bands, classes=classes)
                
                # Prevent Double-Dipping: If using WeightedRandomSampler (is_lazy), data is already 50/50.
                pytorch_weights = None if is_lazy else c_weights
                
                acc, kap, preds, _, _ = train_pytorch_model(net, train_loader, test_loader, epochs=epochs, patience=patience, class_weights=pytorch_weights)
                
            results[m]['acc'].append(acc)
            results[m]['kappa'].append(kap)
            print(f"{m:10s} -> Acc: {acc:.4f} | Kappa: {kap:.4f}")
            
            
    # Aggregate Results
    print("\n" + "="*50)
    print(f"FINAL {num_seeds}-SEED RESULTS ({dataset_name})")
    print("="*50)
    final_df = []
    for m in models_to_test:
        mean_acc = np.mean(results[m]['acc']) * 100
        std_acc = np.std(results[m]['acc']) * 100
        mean_kap = np.mean(results[m]['kappa'])
        std_kap = np.std(results[m]['kappa'])
        final_df.append({'Model': m, 'OA (%)': f"{mean_acc:.2f} ± {std_acc:.2f}", 'Kappa': f"{mean_kap:.4f} ± {std_kap:.4f}"})
        
    df = pd.DataFrame(final_df)
    print(df.to_string(index=False))
    
    out_csv = os.path.join(OUTPUT_DIR, f"Table2_{num_seeds}Seed_Results_{dataset_name.replace(' ', '_')}.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n✅ Results saved to {out_csv}")
    return results

def execute_ablation_studies(X, y, conditions):
    print("\n⏳ Running Ablation Studies (Removing Attention Block)...")
    moist_indices = np.where(conditions == 1)[0]
    dry_indices = np.where(conditions == 0)[0]
    
    train_moist, test_moist = train_test_split(moist_indices, test_size=0.8, random_state=42)
    train_idx = np.concatenate([dry_indices, train_moist])
    test_idx = test_moist
    
    train_loader = DataLoader(SoilDataset(X, y, train_idx), batch_size=32, shuffle=True)
    test_loader = DataLoader(SoilDataset(X, y, test_idx), batch_size=32, shuffle=False)
    
    print("Training ViT WITH Attention...")
    acc_att, _, _, _, _ = train_pytorch_model(ViT1D(), train_loader, test_loader, epochs=15)
    
    print("Training ViT WITHOUT Attention...")
    acc_no_att, _, _, _, _ = train_pytorch_model(ViT_No_Attention(), train_loader, test_loader, epochs=15)
    
    
    print("\n" + "="*50)
    print("TABLE 3: ABLATION RESULTS")
    print(f"ViT (With Attention):    {acc_att*100:.2f}% Accuracy")
    print(f"ViT (Without Attention): {acc_no_att*100:.2f}% Accuracy")
    print(f"Drop without Attention:  {(acc_att - acc_no_att)*100:.2f}%")
    print("="*50)

print("✅ Cell 4 Complete: 10-seed training loop defined.")


✅ Cell 4 Complete: 10-seed training loop defined.


In [5]:
# ==========================================
# CELL 5: EXPLAINABLE AI, PLOTS, & MAIN EXECUTION
# ==========================================
import joblib

def plot_confusion_matrices(y_true, preds_dict):
    """Generates side-by-side confusion matrices for the test set."""
    fig, axes = plt.subplots(1, len(preds_dict), figsize=(5 * len(preds_dict), 5))
    classes = ['Black', 'Red', 'Yellow']
    
    for ax, (model_name, preds) in zip(axes, preds_dict.items()):
        cm = confusion_matrix(y_true, preds)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
        ax.set_title(f'{model_name} Confusion Matrix\n(Tested on Wet Soil)')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        
    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "Figure1_Confusion_Matrices.png")
    plt.savefig(out_path, dpi=300)
    print(f"✅ Confusion matrices saved to {out_path}")
    plt.close()

def run_shap_analysis(model, X_test, model_name="ViT"):
    """Runs SHAP to prove the model is ignoring water bands (1400/1900nm)."""
    print(f"⏳ Running Explainable AI (SHAP) on {model_name}...")
    
    # Flatten spatial dims to get pure spectral signature (B, Bands)
    b, h, w, c = X_test.shape
    X_test_flat = X_test[:, h//2, w//2, :]
    
    # Use SHAP GradientExplainer or DeepExplainer
    # For PyTorch, we need a wrapper function that takes flat tensor and reshapes it if necessary
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    class SHAPWrapper(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
        def forward(self, x_flat):
            # Reconstruct (B, H, W, C) for the model's expected input
            # Use .repeat() instead of .expand() to physically allocate memory for SHAP additivity
            x_recon = x_flat.unsqueeze(1).unsqueeze(1).repeat(1, 15, 15, 1)
            return self.base_model(x_recon)
            
    wrapper = SHAPWrapper(model).to(device).eval()
    
    # Take a small background sample for SHAP
    background = torch.tensor(X_test_flat[:100], dtype=torch.float32).to(device)
    test_samples = torch.tensor(X_test_flat[100:110], dtype=torch.float32).to(device)
    # Use SHAP GradientExplainer to bypass the PyTorch DeepLIFT additivity bug for 3D Convolutions
    e = shap.GradientExplainer(wrapper, background)
    shap_values = e.shap_values(test_samples)
    
    # Plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, test_samples.cpu().numpy(), plot_type="bar", show=False)
    plt.title("SHAP Feature Importance (Spectral Bands)\nProving Mineral Focus vs Water Avoidance")
    out_path = os.path.join(OUTPUT_DIR, "Figure2_SHAP_Importance.png")
    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    print(f"✅ SHAP plot saved to {out_path}")
    plt.close()

def plot_training_curves(losses, model_name="ViT"):
    """Plots and saves the training loss curve."""
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker='o', linestyle='-', color='b')
    plt.title(f"{model_name} Training Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Cross Entropy Loss")
    plt.grid(True)
    out_path = os.path.join(OUTPUT_DIR, f"Figure3_{model_name}_Training_Curve.png")
    plt.savefig(out_path, dpi=300)
    print(f"✅ Training curve saved to {out_path}")
    plt.close()

def generate_false_color_map(model, img_cube, patch_size=15):
    """Feeds an entire .bil cube into the model and generates a spatial prediction map."""
    print("⏳ Generating False Color Spatial Map (Figure 4)...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device).eval()
    h, w, c = img_cube.shape
    
    # We will reconstruct a map of size (h//patch_size, w//patch_size)
    pred_map = np.zeros((h // patch_size, w // patch_size))
    
    # For speed in visualization, we just take the center pixel of patches
    with torch.no_grad():
        for i, row in enumerate(range(0, h - patch_size, patch_size)):
            for j, col in enumerate(range(0, w - patch_size, patch_size)):
                patch = img_cube[row:row+patch_size, col:col+patch_size, :]
                patch_tensor = torch.tensor(patch, dtype=torch.float32).unsqueeze(0).to(device)
                
                # Normalize exactly as SoilDataset does
                mean = patch_tensor.mean(dim=(1,2,3), keepdim=True)
                std = patch_tensor.std(dim=(1,2,3), keepdim=True) + 1e-8
                patch_tensor = (patch_tensor - mean) / std
                
                out = model(patch_tensor)
                pred = torch.argmax(out, dim=1).item()
                pred_map[i, j] = pred
                
    # Plot spatial map
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['black', 'red', 'yellow'])
    plt.figure(figsize=(8, 6))
    plt.imshow(pred_map, cmap=cmap)
    plt.colorbar(ticks=[0, 1, 2], format=plt.FuncFormatter(lambda val, loc: ['Black', 'Red', 'Yellow'][int(val)]))
    plt.title("Spatial False Color Classification Map")
    out_path = os.path.join(OUTPUT_DIR, "Figure4_FalseColor_Map.png")
    plt.savefig(out_path, dpi=300)
    print(f"✅ False Color Map saved to {out_path}")
    plt.close()

# ==========================================
# MAIN KAGGLE EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("🚀 STARTING FULL PIPELINE EXECUTION")
    print("="*50)
    
    # 1. Load Data
    # Dynamically detect if we are on Kaggle or Local PC
    if os.path.exists("/kaggle/input"):
        base_dir = glob.glob("/kaggle/input/**/drive-download*", recursive=True)
        base_dir = base_dir[0] if base_dir else "/kaggle/input"
    else:
        base_dir = r"C:\Users\devgu\Downloads\research_paper_work_3\drive-download-20260618T085601Z-3-002"
    
    try:
        X, y, cond = load_local_dataset(base_dir)
        public_data = load_public_datasets()
        
        # 2. Run 10-Seed Training Pipeline on Local Dataset (BYPASSED)
        # bands_local = X.shape[-1]
        # classes_local = len(np.unique(y))
        # results = execute_10_seed_training(X, y, cond, dataset_name="Local Soil", bands=bands_local, classes=classes_local)
        
        # 2.5 Run Ablation Studies (BYPASSED)
        # execute_ablation_studies(X, y, cond)
        
        # 3. Generate Final Plots for the last seed
        moist_indices = np.where(cond == 1)[0]
        dry_indices = np.where(cond == 0)[0]
        train_moist, test_moist = train_test_split(moist_indices, test_size=0.8, random_state=9)
        train_idx = np.concatenate([dry_indices, train_moist])
        
        print("\n⏳ Re-Training 3D-CNN locally for SHAP and Maps (Fast)...")
        bands_local = X.shape[-1]
        classes_local = len(np.unique(y))
        
        cnn3d = CNN3D(bands=bands_local, classes=classes_local)
        train_loader = DataLoader(SoilDataset(X, y, train_idx), batch_size=32, shuffle=True)
        test_loader = DataLoader(SoilDataset(X, y, test_moist), batch_size=32, shuffle=False)
        _, _, preds_cnn, trained_cnn, losses = train_pytorch_model(cnn3d, train_loader, test_loader, epochs=10)
        
        # Run XAI (SHAP)
        X_test = X[test_moist]
        run_shap_analysis(trained_cnn, X_test, model_name="3D-CNN")
        
        # Generate False Color Map on a single sample image
        sample_bil_path = glob.glob(os.path.join(base_dir, "**", "*.bil"), recursive=True)[0]
        sample_cube = read_bil_file(sample_bil_path)
        if sample_cube is not None:
            generate_false_color_map(trained_cnn, sample_cube)
        
        print("\n🎉 LOCAL SHAP & MAPS GENERATED! Moving to Public Datasets...")
        
        # 4. Evaluate on Tinto Dataset (Skipping HyBEAR as it requires complex spatial masking)
        if 'Tinto' in public_data:
            print("\n🚀 Isolating Tinto Dataset for 10-Seed Evaluation...")
            p_data = public_data['Tinto']
            lazy_ds = p_data['LazyDataset']
            x0, _ = lazy_ds[0]
            b_bands = x0.shape[-1]
            b_classes = len(np.unique(lazy_ds.labels)) if hasattr(lazy_ds, 'labels') and lazy_ds.labels is not None else 3
            execute_10_seed_training(lazy_ds, dataset_name="Tinto", bands=b_bands, classes=b_classes)
            
        print("\n🎉 TINTO EVALUATION COMPLETE! All assets saved to /kaggle/working/")
        
    except Exception as e:
        print(f"\n❌ Pipeline failed. Did you update the 'base_dir' path? Error: {e}")



🚀 STARTING FULL PIPELINE EXECUTION


⏳ Extracting Local Dataset from: /kaggle/input
Found 47 .bil files. Extracting patches...


✅ Local Data Loaded: 23961 patches extracted. Shape: (23961, 15, 15, 168)
⏳ Loading REAL Public Datasets (HyBEAR, KarLy, Shangyu, Tinto)...


⏳ Building robust image index for Kaggle filesystem...
✅ Fast index built! Found 500 physical images.
🧹 Metadata Cleaned: Synced 250 physical files out of 1954 total CSV rows!
✅ HyBEAR LazyLoader Initialized
✅ KarLy Loaded: (679, 125)
✅ Indian Pines Loaded: (10249, 200)


✅ Tinto LazyLoader Initialized

⏳ Re-Training 3D-CNN locally for SHAP and Maps (Fast)...


...

      Early stopping triggered at epoch 38


1D-CNN     -> Acc: 0.8752 | Kappa: 0.7918


      Early stopping triggered at epoch 57


GCN        -> Acc: 0.8792 | Kappa: 0.7985


      Early stopping triggered at epoch 35
